In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.mart.fct_orders_flat AS
SELECT
  -- Grain identifier
  o.order_id,
  o.order_date,
  o.order_status,
  CASE WHEN o.order_status = 'Completed' THEN TRUE ELSE FALSE END AS is_completed,
  o.shipping_method,

  -- Order-level money fields (safe here — this is the correct grain for them)
  o.discount_amount,
  o.shipping_cost,
  o.tax_amount,
  o.order_total,

  -- Rolled-up item-level metrics (aggregated from order_items, one row per order)
  oi_agg.item_count,
  oi_agg.total_units,
  oi_agg.total_item_revenue,
  oi_agg.total_cogs,
  oi_agg.total_gross_profit,

  -- Rolled-up return metrics (aggregated from returns, one row per order)
  CASE WHEN ret_agg.order_id IS NOT NULL THEN TRUE ELSE FALSE END AS has_return,
  COALESCE(ret_agg.return_count, 0)          AS return_count,
  COALESCE(ret_agg.total_refund_amount, 0)   AS total_refund_amount,

  -- Customer dimension
  c.customer_id,
  c.customer_segment,
  c.acquisition_channel,
  c.gender,
  c.city               AS customer_city,
  c.signup_date,

  -- Customer home region
  cust_r.region_id      AS customer_region_id,
  cust_r.region_name    AS customer_region_name,
  cust_r.market_type    AS customer_market_type,

  -- Shipping / fulfillment side
  o.region_id           AS shipping_region_id,
  ship_r.region_name    AS shipping_region_name,
  ship_r.market_type    AS shipping_market_type,
  s.store_id,
  s.store_name,
  s.store_type,

  -- Date dimension
  d.year,
  d.quarter,
  d.month,
  d.month_name,
  d.year_month,
  d.week,
  d.day_name,
  d.is_weekend,
  d.is_holiday,
  d.holiday_name

FROM ecommerce.clean.orders  o
JOIN ecommerce.clean.customers c ON o.customer_id = c.customer_id
LEFT JOIN ecommerce.base.regions cust_r ON c.region_id = cust_r.region_id
LEFT JOIN ecommerce.base.regions ship_r ON o.region_id = ship_r.region_id
LEFT JOIN ecommerce.base.stores  s      ON o.store_id  = s.store_id
LEFT JOIN ecommerce.base.date    d      ON o.order_date = d.date
LEFT JOIN (
  SELECT
    order_id,
    COUNT(*)              AS item_count,
    SUM(quantity)          AS total_units,
    SUM(item_revenue)      AS total_item_revenue,
    SUM(total_cost)        AS total_cogs,
    SUM(gross_profit)      AS total_gross_profit
  FROM ecommerce.clean.order_items
  GROUP BY order_id
) oi_agg ON o.order_id = oi_agg.order_id
LEFT JOIN (
  SELECT
    order_id,
    COUNT(*)              AS return_count,
    SUM(refund_amount)     AS total_refund_amount
  FROM ecommerce.clean.returns
  GROUP BY order_id
) ret_agg ON o.order_id = ret_agg.order_id;

### TEST 1 — Row count parity (mart should have exactly one row per order)

In [0]:
%sql
-- Expect: source_row_count = mart_row_count, row_diff = 0
SELECT
  (SELECT COUNT(*) FROM ecommerce.clean.orders) AS source_row_count,
  (SELECT COUNT(*) FROM ecommerce.mart.fct_orders_flat) AS mart_row_count,
  (SELECT COUNT(*) FROM ecommerce.clean.orders)
    - (SELECT COUNT(*) FROM ecommerce.mart.fct_orders_flat) AS row_diff;

### TEST 2 — Grain check (order_id must be unique)

In [0]:
%sql
-- Expect: 0 rows
SELECT order_id, COUNT(*) AS occurrences
FROM ecommerce.mart.fct_orders_flat
GROUP BY order_id
HAVING COUNT(*) > 1;

### TEST 3 — No fan-out risk from the order_items and returns aggregation subqueries

In [0]:
%sql
-- Expect: 0 rows for both — each subquery is pre-aggregated to one row per order_id
SELECT 'order_items_agg' AS agg_source, order_id, COUNT(*) AS occurrences
FROM (
  SELECT order_id, COUNT(*) AS c
  FROM ecommerce.clean.order_items
  GROUP BY order_id
) t GROUP BY order_id HAVING COUNT(*) > 1
UNION ALL
SELECT 'returns_agg', order_id, COUNT(*)
FROM (
  SELECT order_id, COUNT(*) AS c
  FROM ecommerce.clean.returns
  GROUP BY order_id
) t GROUP BY order_id HAVING COUNT(*) > 1;

### TEST 4 — No orphaned rows dropped (every clean.orders row should appear in the mart)

In [0]:
%sql
-- Expect: 0 rows
SELECT o.order_id
FROM ecommerce.clean.orders o
LEFT JOIN ecommerce.mart.fct_orders_flat f ON o.order_id = f.order_id
WHERE f.order_id IS NULL;

### TEST 5 — is_completed flag logic (should only be TRUE when order_status = 'Completed')

In [0]:
%sql
-- Expect: 0 rows
SELECT order_status, is_completed, COUNT(*) AS occurrences
FROM ecommerce.mart.fct_orders_flat
GROUP BY order_status, is_completed
HAVING (order_status = 'Completed' AND is_completed = FALSE)
    OR (order_status != 'Completed' AND is_completed = TRUE);

### TEST 6 — Every order should have at least one line item (item_count and total_units should never be null/zero)

In [0]:
%sql
-- Expect: 0 rows. If any appear, investigate whether these are legitimately item-less orders
-- (e.g. Cancelled before any line item was added) or a join bug.
SELECT order_id, order_status, item_count, total_units
FROM ecommerce.mart.fct_orders_flat
WHERE item_count IS NULL OR item_count = 0
   OR total_units IS NULL OR total_units = 0;

### TEST 7 — total_item_revenue reconciliation (mart rollup vs direct source aggregation)

In [0]:
%sql
-- Expect: revenue_diff = 0 (or negligible rounding, e.g. < 0.01)
SELECT
  (SELECT ROUND(SUM(item_revenue), 2) FROM ecommerce.clean.order_items) AS source_revenue,
  (SELECT ROUND(SUM(total_item_revenue), 2) FROM ecommerce.mart.fct_orders_flat) AS mart_revenue,
  (SELECT ROUND(SUM(item_revenue), 2) FROM ecommerce.clean.order_items)
    - (SELECT ROUND(SUM(total_item_revenue), 2) FROM ecommerce.mart.fct_orders_flat) AS revenue_diff;

### TEST 8 — order_total reconciliation (total_item_revenue + tax_amount + shipping_cost ≈ order_total)

In [0]:
%sql
-- Expect: 0 rows (small rounding tolerance of 0.01 applied)
-- Per the data dictionary: order_total = item revenue (post-discount) + tax_amount + shipping_cost
SELECT
  order_id,
  total_item_revenue,
  tax_amount,
  shipping_cost,
  order_total,
  ROUND(total_item_revenue + tax_amount + shipping_cost, 2) AS calculated_total,
  ROUND(order_total - (total_item_revenue + tax_amount + shipping_cost), 2) AS diff
FROM ecommerce.mart.fct_orders_flat
WHERE ABS(order_total - (total_item_revenue + tax_amount + shipping_cost)) > 0.01;

### TEST 9 — gross profit reconciliation (total_gross_profit vs direct source aggregation)

In [0]:
%sql
-- Expect: gp_diff = 0
SELECT
  (SELECT ROUND(SUM(gross_profit), 2) FROM ecommerce.clean.order_items) AS source_gp,
  (SELECT ROUND(SUM(total_gross_profit), 2) FROM ecommerce.mart.fct_orders_flat) AS mart_gp,
  (SELECT ROUND(SUM(gross_profit), 2) FROM ecommerce.clean.order_items)
    - (SELECT ROUND(SUM(total_gross_profit), 2) FROM ecommerce.mart.fct_orders_flat) AS gp_diff;

### TEST 10 — has_return flag logic (should only be TRUE when return_count > 0)

In [0]:
%sql
-- Expect: 0 rows
SELECT has_return, return_count, COUNT(*) AS occurrences
FROM ecommerce.mart.fct_orders_flat
GROUP BY has_return, return_count
HAVING (has_return = TRUE AND return_count = 0)
    OR (has_return = FALSE AND return_count > 0);

### TEST 11 — total_refund_amount reconciliation (mart rollup vs direct source aggregation)

In [0]:
%sql
-- Expect: refund_diff = 0
SELECT
  (SELECT ROUND(SUM(refund_amount), 2) FROM ecommerce.clean.returns) AS source_refunds,
  (SELECT ROUND(SUM(total_refund_amount), 2) FROM ecommerce.mart.fct_orders_flat) AS mart_refunds,
  (SELECT ROUND(SUM(refund_amount), 2) FROM ecommerce.clean.returns)
    - (SELECT ROUND(SUM(total_refund_amount), 2) FROM ecommerce.mart.fct_orders_flat) AS refund_diff;

### TEST 12 — Refund sanity (total_refund_amount should never exceed order_total)

In [0]:
%sql
-- Expect: 0 rows. Any result flags an over-refund and needs investigation upstream.
SELECT order_id, order_total, total_refund_amount
FROM ecommerce.mart.fct_orders_flat
WHERE total_refund_amount > order_total;

### TEST 13 — Excluded fields truly absent (item-grain fields should NOT be in this table)

In [0]:
%sql
-- Expect: query FAILS (column not found) — that failure is the pass condition
SELECT order_item_id, product_id, unit_price FROM ecommerce.mart.fct_orders_flat LIMIT 1;

### TEST 14 — Null profiling on key columns

In [0]:
%sql
-- Expect: 0 for customer_segment/order_total/item_count; some nulls in shipping_region/store/date
-- are acceptable (e.g. Pending/Cancelled orders may lack store_id) — eyeball, don't hard-fail
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(customer_segment)     AS null_segment,
  COUNT(*) - COUNT(order_total)          AS null_order_total,
  COUNT(*) - COUNT(item_count)           AS null_item_count,
  COUNT(*) - COUNT(shipping_region_id)   AS null_shipping_region,
  COUNT(*) - COUNT(store_id)             AS null_store,
  COUNT(*) - COUNT(year)                 AS null_date_join
FROM ecommerce.mart.fct_orders_flat;

### TEST 15 — Referential sanity (customer_id resolves to clean.customers)

In [0]:
%sql
-- Expect: 0 rows
SELECT f.customer_id
FROM ecommerce.mart.fct_orders_flat f
LEFT JOIN ecommerce.clean.customers c ON f.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

### TEST 16 — Value range sanity on money fields (none should be negative)

In [0]:
%sql
-- Expect: 0 rows
SELECT *
FROM ecommerce.mart.fct_orders_flat
WHERE order_total < 0
   OR discount_amount < 0
   OR shipping_cost < 0
   OR tax_amount < 0
   OR total_refund_amount < 0
   OR total_item_revenue < 0;

### TEST 17 — Date join sanity (order_date should match the joined date-dimension row exactly)

In [0]:
%sql
-- Expect: 0 rows
SELECT f.order_id, f.order_date, d.date AS joined_date
FROM ecommerce.mart.fct_orders_flat f
LEFT JOIN ecommerce.base.date d ON f.order_date = d.date
WHERE d.date IS NULL OR f.order_date != d.date;

### TEST 18 — Distinct value spot-check on order_status

In [0]:
%sql
-- Expect: only 'Completed', 'Cancelled', 'Pending', 'Returned'
SELECT DISTINCT order_status FROM ecommerce.mart.fct_orders_flat;